## 1.Import Necessary Libraries

In [1]:
import pandas as pd
import keras
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

## 2.Import Dataset

In [2]:
diabetes_data = pd.read_csv('diabetes.csv')
diabetes_data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


## 3.Data Preparation

In [3]:
X = diabetes_data.drop('Outcome',axis=1)
y = diabetes_data.Outcome

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=46,stratify=y)

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

### HYPERPARAMETER TUNING

#### List the Hyperparamaters we used so far....
1. Learning Rate
2. Epochs
3. Batches
4. Activation Functions
5. Optimizers
6. Loss Functions
7. Kernel Initializers

### ========================================================

In [6]:
!pip install scikeras

Defaulting to user installation because normal site-packages is not writeable


## Hyperparameter all at once
#### This process is computationally expensive.|

In [7]:
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout

from scikeras.wrappers import KerasClassifier
##I am building a neural network model using Keras Functional API,
# and I am using GridSearchCV with KerasClassifier to find the best 
# parameters for the model
from sklearn.model_selection import GridSearchCV,KFold

In [22]:
# def create_model(learning_rate=0.001,dropout_rate=0.0,activation_function='relu',init='glorot_uniform',neuron1=8,neuron2=4):
#     model = Sequential()
#     model.add(Dense(neuron1,input_shape=(8,),kernel_initializer=init,activation=activation_function))
#     model.add(Dropout(dropout_rate))
#     model.add(Dense(neuron2,kernel_initializer=init,activation=activation_function))
#     model.add(Dropout(dropout_rate))
#     model.add(Dense(1,activation='sigmoid'))
    
#     optimizer = keras.Optimizer='adam'(learning_rate=learning_rate)
# #     model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])
#     return model

In [23]:
def create_model(learning_rate=0.001,dropout_rate=0.0,activation_function='relu', init='glorot_uniform',neuron1=8, neuron2=4):

    model = Sequential()
    model.add(Dense(neuron1,input_shape=(8,),kernel_initializer=init,activation=activation_function))
    model.add(Dropout(dropout_rate))
    model.add(Dense(neuron2,kernel_initializer=init,activation=activation_function))    
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])
    return model

In [24]:
model = KerasClassifier(model = create_model,verbose=1)

In [18]:
# param_grid ={'model__learning_rate':[0.001,0.01],
#              #if its model_lr = Just variable naming, but here it is model__learning_rate -> Navigate inside object
#              'model__dropout_rate':[0.0,0.20],
#              'model__activation_function':['relu','tanh'],
#              'model__init':['glorot_uniform','he_uniform'],
#              'model__neuron1':[8,16],
#              'model__neuron1':[4,8],
#              'batch_size':[10,20],
#              'epochs':[10,50]
#             }

In [25]:
param_grid = {'model__learning_rate': [0.001, 0.01], 
               #if its model_lr = Just variable naming, but here it is model__learning_rate -> Navigate inside object
              'model__dropout_rate': [0.0, 0.2],
              'model__activation_function': ['relu', 'tanh'],
              'model__init': ['glorot_uniform', 'he_uniform'],
              'model__neuron1': [8, 16],
              'model__neuron2': [4, 8],
              'batch_size': [10, 20],
              'epochs': [10, 50]
             }

In [28]:
kfold = KFold(n_splits=3,shuffle=True,random_state=46)
grid = GridSearchCV(estimator=model,param_grid=param_grid,cv=kfold,verbose=10,n_jobs=1)

In [ ]:
grid_result = grid.fit(X_train,y_train)

In [ ]:
print("Best: %f using %s" %(grid_result.beat_score_,grid_result.best_params_))

means  = grid_result.cv_results_["mean_test_score"]
stds   = grid_result.cv_results_["std_test_score"]
params = grid_result.cv_results_['params']


for mean,std,prams in zip(means,stds,params):
    print("%f (%f) with: %r" % (mean,std,param))

In [ ]:
results_df = pd.DataFrame(grid_result.cv_results_)
results_df

In [ ]:
results_df.to_csv("results.csv")

# THE END !!!